In [ ]:
import sys
sys.path.append('Track2/Functions')

import os
import json
import anthropic
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from C2_4_Func import PromptRegistry, PromptVersion, ABTest

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',
    temperature=0.3,
    anthropic_api_key=ANTHROPIC_API_KEY
)
parser = StrOutputParser()
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

Settings.llm = LlamaAnthropic(
    model='claude-haiku-4-5-20251001',
    api_key=ANTHROPIC_API_KEY
)
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

from pathlib import Path
from C2_4_Func import build_c2_4_corpus, persist_c2_4_corpus, load_c2_4_corpus

CORPUS_DIR = Path('Track2') / 'demos' / 'data' / 'C2.4_corpus'
raw_docs = build_c2_4_corpus()
persist_c2_4_corpus(raw_docs, CORPUS_DIR)
loaded_docs = load_c2_4_corpus(CORPUS_DIR)

finance_docs = [Document(text=d['text'], metadata=d['metadata']) for d in loaded_docs if d['metadata']['domain'] == 'finance']
education_docs = [Document(text=d['text'], metadata=d['metadata']) for d in loaded_docs if d['metadata']['domain'] == 'education']
all_docs = finance_docs + education_docs

---
## Section 6 · Lab — End-to-End Multi-Step AI Workflow

### Scenario: Investment & Learning Assistant

<small>
A fintech platform serves users who are both **investors** and **learners**.
The assistant must:
1. Answer investment questions using a RAG index over financial documents
2. Recommend courses based on the user's learning history
3. Call live data tools via MCP (stock prices, course enrollment)
4. Manage conversation memory so context accumulates across turns
5. Use versioned prompts so the team can A/B test response quality

**Components used in this lab:**
| Component | Purpose |
|-----------|---------|
| LangChain LCEL chain | Orchestrate prompt → LLM → parser |
| LlamaIndex VectorStoreIndex | RAG over financial documents |
| MCP tools | Live stock prices and enrollment checks |
| Multi-agent coordinator | Route finance vs education sub-tasks |
| PromptRegistry | Version-controlled prompts |
| RunnableWithMessageHistory | Per-user memory |
</small>

### 6.1 Component Setup

In [ ]:
# ── Prompt registry for the lab ──────────────────────────────────────────────
lab_registry = PromptRegistry()

lab_registry.register(PromptVersion(
    name='investment_assistant',
    version='v1',
    template=(
        'You are an investment and learning assistant.\n'
        'Use the retrieved context to answer accurately.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}'
    ),
    author='lab',
))

lab_registry.register(PromptVersion(
    name='investment_assistant',
    version='v2',
    template=(
        'You are a personalised investment and learning assistant for finance professionals.\n'
        'User profile: {user_profile}\n\n'
        'Relevant context:\n{context}\n\n'
        'Respond in two sections:\n'
        '1. **Investment insight** (if relevant)\n'
        '2. **Learning recommendation** (if relevant)\n\n'
        'Question: {question}'
    ),
    author='lab',
))

print('Lab prompt registry ready. Versions:', lab_registry.list_versions('investment_assistant'))

In [ ]:
# ── LlamaIndex RAG component ─────────────────────────────────────────────────

lab_index = VectorStoreIndex.from_documents(all_docs)  # reuse docs from Section 2
lab_qe    = lab_index.as_query_engine(similarity_top_k=3)

def retrieve_context(question: str) -> str:
    """Retrieve relevant document context for a question."""
    result = lab_qe.query(question)
    return str(result)

print('RAG index ready. Testing retrieval...')
ctx = retrieve_context('What are the growth prospects for edtech companies?')
print(ctx[:300])

In [ ]:
# ── LCEL chain with versioned prompt and memory ──────────────────────────────

def build_lab_chain(prompt_version: str = 'v2'):
    pv = lab_registry.get('investment_assistant', prompt_version)
    prompt_template = ChatPromptTemplate.from_messages([
        ('system', pv.template),
        MessagesPlaceholder(variable_name='history'),
        ('human', '{question}'),
    ])
    return prompt_template | llm | parser

lab_session_store = {}

def get_lab_history(session_id: str):
    if session_id not in lab_session_store:
        lab_session_store[session_id] = InMemoryChatMessageHistory()
    return lab_session_store[session_id]

def build_memory_chain(prompt_version: str = 'v2'):
    base_chain = build_lab_chain(prompt_version)
    return RunnableWithMessageHistory(
        base_chain,
        get_lab_history,
        input_messages_key='question',
        history_messages_key='history',
    )

print('Lab chain factory ready.')

### 6.2 Full Workflow Execution
<small>
The assistant receives a user query, retrieves RAG context, passes it through the versioned chain,
and maintains memory across turns.
</small>

In [ ]:
async def run_lab_workflow(session_id: str, user_query: str,
                           user_profile: dict, prompt_version: str = 'v2',
                           use_mcp: bool = True) -> str:
    """
    Full workflow:
    1. Retrieve RAG context (LlamaIndex)
    2. Call MCP tool if live data is needed
    3. Run LCEL chain with versioned prompt + memory
    """
    # Step 1: RAG retrieval
    rag_context = retrieve_context(user_query)

    # Step 2: MCP live data (optional)
    mcp_data = {}
    if use_mcp and any(k in user_query.lower() for k in ['price', 'stock', 'enroll', 'course']):
        try:
            params = StdioServerParameters(command=sys.executable, args=['mcp_c24_server.py'])
            async with stdio_client(params) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    # Pull portfolio prices if tickers mentioned
                    for ticker in ['AAPL', 'JPM', 'UNH']:
                        if ticker in user_query.upper():
                            res = await session.call_tool('get_stock_price', {'ticker': ticker})
                            mcp_data[ticker] = json.loads(res.content[0].text)
        except Exception as e:
            mcp_data['mcp_error'] = str(e)

    # Step 3: Build enriched context
    full_context = rag_context
    if mcp_data:
        full_context += f'\n\nLive market data:\n{json.dumps(mcp_data, indent=2)}'

    # Step 4: LCEL chain with memory
    chain = build_memory_chain(prompt_version)
    response = chain.invoke(
        {
            'question':     user_query,
            'context':      full_context,
            'user_profile': json.dumps(user_profile),
        },
        config={'configurable': {'session_id': session_id}}
    )
    return response

print('Lab workflow function ready.')

In [ ]:
# ── Run the lab workflow ─────────────────────────────────────────────────────

student_investor_profile = {
    'name': 'Priya',
    'role': 'Financial Analyst',
    'completed_courses': ['DS-101', 'FIN-201'],
    'portfolio': ['AAPL', 'JPM', 'UNH'],
    'risk_tolerance': 'moderate',
    'learning_goal': 'Evaluate AI-driven healthcare companies',
}

lab_session = 'priya-session-001'

conversation = [
    'What are the growth prospects for healthcare AI companies in my portfolio?',
    'Which course should I take next to better analyse AI companies?',
    'Based on what we discussed, what is the single most important risk I should monitor?',
]

for turn, query in enumerate(conversation, 1):
    print(f'\n[Turn {turn}] Priya: {query}')
    answer = await run_lab_workflow(
        session_id=lab_session,
        user_query=query,
        user_profile=student_investor_profile,
        prompt_version='v2',
        use_mcp=True,
    )
    print(f'[Turn {turn}] Assistant:\n{answer[:500]}')
    print('-' * 60)

In [ ]:
# ── A/B test the lab prompt versions ────────────────────────────────────────

def score_lab_response(output: str) -> float:
    """Score: rewards structured sections and domain-specific language."""
    score = 0.4
    if 'investment' in output.lower(): score += 0.2
    if 'learning' in output.lower() or 'course' in output.lower(): score += 0.2
    if len(output.split()) > 50:  score += 0.2  # minimum depth
    return round(score, 2)

lab_ab = ABTest(client, lab_registry, 'investment_assistant', 'v1', 'v2', traffic_split=0.5)

test_queries = [
    ('q-001', 'What are the top investment risks for 2024?', {'question': 'What are the top investment risks for 2024?', 'context': 'Healthcare AI market expected 48% CAGR.', 'user_profile': '{}'}),
    ('q-002', 'Which DS course should I take after DS-201?', {'question': 'Which DS course should I take after DS-201?', 'context': 'DS-301 ML Fundamentals requires DS-201.', 'user_profile': '{}'}),
    ('q-003', 'Is EduLearn a good investment?', {'question': 'Is EduLearn a good investment?', 'context': 'EduLearn Q3: revenue +28% YoY, NPS 72.', 'user_profile': '{}'}),
]

print('Running prompt A/B test...\n')
for qid, _, tvars in test_queries:
    result = lab_ab.run(qid, tvars, score_lab_response)
    print(f"  [{qid}] v={result['version']} score={result['score']}")

print('\n=== A/B Test Summary ===')
print(json.dumps(lab_ab.summary(), indent=2))

### 6.3 Lab Extensions

<small>
**Extension 1 — Add a healthcare specialist**
Extend the coordinator in Section 4 with a third specialist that answers clinical questions.
Keep the two-agent rule by making the healthcare specialist a sub-tool of the education specialist.

**Extension 2 — Persistent memory**
Replace `InMemoryChatMessageHistory` with a Redis-backed store using `langchain_community.chat_message_histories.RedisChatMessageHistory`.
Test that memory survives process restarts.

**Extension 3 — Prompt performance dashboard**
Extend `PromptRegistry.compare_versions()` to return confidence intervals (mean ± std).
Add a `PromptRegistry.auto_promote()` method that marks the highest-scoring version as production.

**Extension 4 — LlamaIndex sub-question engine**
Replace the basic VectorStoreIndex query engine with a `SubQuestionQueryEngine` that decomposes
complex questions like 'Compare the growth rates of FinTech Corp and EduLearn Inc.' into sub-queries.

**Extension 5 — MCP server with authentication**
Add an API key header check to the MCP server using the `httpx` transport instead of stdio.
Demonstrate that unauthenticated clients receive an error.
</small>